Experiment D2 Reproduction


Google drive setup, package installation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/ConceptualizingConceptDrift')

In [ ]:
!pip install xplique timm opencv-python

In [ ]:
import shutil
shutil.copytree(
    '/content/drive/MyDrive/data/imageNet/imagenet_images',
    '/content/imagenet_images'
)

'/content/imagenet_images'

Loading and preprocessing images and models

In [ ]:
from datasets import load_dataset
from PIL import Image
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from concept_helpers.DeepView_Craft import CraftTorchDV as Craft
from concept_helpers.DeepView_Craft import CraftTorchSupervised as CraftS
from concept_helpers.combined_crafts import CombinedCrafts

import urllib.request
import glob
import torch
import torch.nn as nn
from torchvision import transforms
import timm

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from scipy.sparse.linalg import eigs
from sklearn.ensemble import  RandomForestClassifier

import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import NMF
from sklearn.metrics import accuracy_score
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

import random

from xplique.concepts.craft import BaseCraft, DisplayImportancesOrder, Factorization, Sensitivity
from sklearn.decomposition import non_negative_factorization
from experiment_helpers.helper_function import *
from experiment_helpers.driftLocalizer import Localizer

import os

device = 'cuda'

# loading any timm model
model = timm.create_model('nf_resnet50.ra2_in1k', pretrained=True)
model = model.to(device)

# processing
config = resolve_data_config({}, model=model)
transform = create_transform(**config)
to_pil = transforms.ToPILImage()

# cut the model in twop arts (as explained in the paper)
# first part is g(.) our 'input_to_latent' model, second part is h(.) our 'latent_to_logit' model
g = nn.Sequential(*(list(model.children())[:4]))  # input to penultimate layer
h = nn.Sequential(*(list(model.children())[4:]))  # penultimate layer to logits


with urllib.request.urlopen('https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt') as f:
        imagenet_class_names = np.array(f.read().decode('utf-8').split('\n'))

def gen_images(filelist,folder_names,folder_name2class_id):
        for f in filelist:
            folder_name = f.split('/')[-2]
            if folder_name in folder_names:
                class_id = folder_name2class_id[folder_name]
                im = Image.open(f)
                if len(im.getbands()) == 3:
                    yield np.array(im.resize((224, 224))), class_id

# idd_folder = 'path/to/subset/of/imagenet'
idd_folder = '/content/imagenet_images'


# idd_folder_names = os.listdir('path/to/subset/of/imagenet')
idd_folder_names = os.listdir(idd_folder)
idd_class_names = idd_folder_names

idd_class_ids = [np.where(imagenet_class_names == class_name)[0][0] for class_name in idd_class_names]
folder_name2class_id = dict(zip(idd_folder_names, idd_class_ids))
filelist = glob.glob(f'{idd_folder}/*/*.jpg')


images, labels = zip(*gen_images(filelist,idd_folder_names,folder_name2class_id))
images, labels = np.array(images), np.array(labels)
preprocessed_images = torch.stack([transform(to_pil(img)) for img in images], 0)
print(preprocessed_images.shape)

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

torch.Size([3000, 3, 256, 256])


## Reproduction
50 runs of:
1. imitating abrupt drift with random drift labels per class
2. embedding full size images through foundation model
3. train drift localizer on embeddings and drift labels
4. fitting NMF per drift phase on patches to extract concepts into concept bank V_drift
5. representing activations according to basis V_drift
6. measure global, phase and local importances for concepts
7. running different h_tilde variations

In [ ]:
import random


label_maps = []
drift_ratios = []
drift_localizer = []
drift_comparison = []
drift_forest = []


one_local_one_global = []
one_local = []
two_local = []
three_local = []
one_global = []
two_global = []
three_global = []

one_local_one_global_preds = []
one_local_preds = []
two_local_preds = []
three_local_preds = []
one_global_preds = []
two_global_preds = []
three_global_preds = []

one_local_one_global_l = []
one_local_l = []
two_local_l = []
three_local_l = []
one_global_l = []
two_global_l = []
three_global_l = []

one_local_one_global_preds_l = []
one_local_preds_l = []
two_local_preds_l = []
three_local_preds_l = []
one_global_preds_l = []
two_global_preds_l = []
three_global_preds_l = []

one_local_l_probs = []
two_local_l_probs = []
three_local_l_probs = []
one_local_preds_l_probs = []
two_local_preds_l_probs = []
three_local_preds_l_probs = []

reconstructed_single_concepts = []
reconstructed_single_concepts_preds = []
reconstructed_2_concepts = []
reconstructed_2_concepts_preds = []
reconstructed_3_concepts = []
reconstructed_3_concepts_preds = []
reconstructed_all_concepts = []
reconstructed_all_concepts_preds = []

# for j in range(50):
for j in range(50):

    sample_ids = np.random.choice(len(preprocessed_images),500, False)

    sample_images = preprocessed_images[sample_ids]


    # Map each digit to a label indicating whether it occurs before or after the change point, or both, or neither
    #  0 - never, 1 - before, 2 - after, 3 - both


    # Initialize the label_map keys
    keys = idd_class_ids

    # Shuffle the keys for more randomness
    random.shuffle(keys)

    # Assign at least one of each label (0, 1, 2)
    initial_labels = [0, 1, 2]
    random.shuffle(initial_labels)

    # Ensure that the first three keys have 0, 1, and 2 respectively
    label_map = {keys[i]: initial_labels[i] for i in range(3)}

    # Randomly assign labels for the remaining keys
    for i in range(3, len(keys)):
        label_map[keys[i]] = random.randint(0, 2)

    label_maps.append(label_map)

    labels_mapped = np.array([label_map[class_id] for class_id in labels])

    drift_labels = labels_mapped[sample_ids]

    # Randomly assign labels of 1 or 2 to samples with label 3
    #  (i.e., digits that occur both before and after the change point)
    label_2_idx = np.where(drift_labels == 2)[0]
    y_mixed = drift_labels.copy()
    y_mixed[label_2_idx] = np.random.choice([0, 1], size=len(label_2_idx))

    sample_labels = y_mixed

    drift_ratios.append({"BD": len(np.where(drift_labels == 0)[0]),
                         "AD": len(np.where(drift_labels == 1)[0]),
                         "Both": len(np.where(drift_labels == 2)[0])})

    full_size = 256
    patch_size= 100


    #Supervised CRAFT Training
    h_craftdv = CraftS(input_to_latent_model=g,
                        latent_to_logit_model=h,
                        number_of_concepts=5,
                        inputs=sample_images,
                        labels=sample_labels,
                        batch_size=64,
                        patch_size=full_size,
                        device=device)

    patches, patch_act, train_labels = h_craftdv._extract_patches(sample_images, sample_labels )

    bd_indices = np.where(sample_labels != 1)[0]
    ad_indices = np.where(sample_labels != 0)[0]

    bd_fit = Craft(input_to_latent_model=g,
                    latent_to_logit_model=h,
                    number_of_concepts=10,
                    # labels=h_y,
                    patch_size=patch_size,
                    batch_size=64,
                    device=device)
    print("Fitting Unsupervised Craft....")
    bd_crops, bd_crops_u, bd_w = bd_fit.fit(sample_images[bd_indices])


    ad_fit = Craft(input_to_latent_model=g,
                        latent_to_logit_model=h,
                        number_of_concepts=10,
                        # labels=h_y,
                        patch_size=patch_size,
                        batch_size=64,
                        device=device)
    print("Fitting Unsupervised Craft....")
    ad_crops, ad_crops_u, ad_w = ad_fit.fit(sample_images[ad_indices])

    drift_basis = np.vstack([bd_w, ad_w])

    drift_craft = CombinedCrafts(input_to_latent_model=g,
                    latent_to_logit_model=h,
                    number_of_concepts=len(drift_basis),
                    inputs=sample_images,
                    labels=sample_labels,
                    basis = drift_basis,
                    batch_size=64,
                    patch_size=patch_size,
                    device=device)
    print("Fitting Craft....")
    drift_craft.transform_all()


    X_clean = patch_act
    y_clean = train_labels

    # Initialize a random forest model with max_leaf_nodes=150
    localizer_model = Localizer()


    # Perform the train-test split on X_clean and sample_labels
    X_train_clean, X_test_clean, y_train, y_test = \
        train_test_split(X_clean, y_clean, train_size=0.7, random_state=42)

    # Fit the model to the mixed set (group 3 is randomly assigned to 1 or 2)
    print('Fitting Random Forest classifier...')
    localizer_model.fit(X_train_clean, y_train);
    print('Fitting complete.')

    localizer_bin_preds = localizer_model.l_predict(X_test_clean)
    drift_localizer.append(accuracy_score(localizer_bin_preds, y_test))

    drift_imp = np.round(estimate_importance_l(localizer_model, drift_craft, drift_basis, X_train_clean),3)



    # y_preds_l, _ = compute_predictions(localizer_model,X_test_clean)
    image_drift_imp_l = [estimate_importance_helper_l(drift_craft,localizer_model,drift_basis,
                                                  image,class_of_interest=localizer_bin_preds[i])
                               for i,image in enumerate(X_test_clean)]



    one_local_one_global_l.append(local_one_imp_concept_globally_l(drift_craft,image_drift_imp_l,y_test))
    one_local_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=1,labels=y_test))
    two_local_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=2,labels=y_test))
    three_local_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=3,labels=y_test))
    # all_local_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=20,labels=y_test))

    one_global_l.append(global_imp_concepts_locally_l(drift_craft,image_drift_imp_l,num=1,labels=y_test))
    two_global_l.append(global_imp_concepts_locally_l(drift_craft,image_drift_imp_l,num=2,labels=y_test))
    three_global_l.append(global_imp_concepts_locally_l(drift_craft,image_drift_imp_l,num=3,labels=y_test))
    # all_global_l.append(global_imp_concepts_locally_l(drift_craft,image_drift_imp_l,num=20,labels=y_test))

    # all_local_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=20,labels=y_test))


    one_local_one_global_preds_l.append(local_one_imp_concept_globally_l(drift_craft,image_drift_imp_l,localizer_bin_preds))
    one_local_preds_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=1,labels=localizer_bin_preds))
    two_local_preds_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=2,labels=localizer_bin_preds))
    three_local_preds_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=3,labels=localizer_bin_preds))


    # all_local_preds_l.append(local_imp_concepts_globally_l(drift_craft,image_drift_imp_l,num=20,labels=localizer_bin_preds))

    one_global_preds_l.append(global_imp_concepts_locally_l(drift_craft,image_drift_imp_l,num=1,labels=localizer_bin_preds))
    two_global_preds_l.append(global_imp_concepts_locally_l(drift_craft,image_drift_imp_l,num=2,labels=localizer_bin_preds))
    three_global_preds_l.append(global_imp_concepts_locally_l(drift_craft,image_drift_imp_l,num=3,labels=localizer_bin_preds))
    # all_global_preds_l.append(global_imp_concepts_locally_l(drift_craft,image_drift_imp_l,num=20,labels=localizer_bin_preds))


    localizer_bin_train_preds = localizer_model.l_predict(X_train_clean)
    image_drift_imp_l_train = [estimate_importance_helper_l(drift_craft,localizer_model,drift_basis,
                                                  image,class_of_interest=localizer_bin_train_preds[i])
                               for i,image in enumerate(X_train_clean)]
    concept_dist = concept_counter(image_drift_imp_l_train,localizer_bin_train_preds)

    one_local_l_probs.append(local_imp_concepts_probability(concept_dist,image_drift_imp_l,num=1,labels=y_test))
    two_local_l_probs.append(local_imp_concepts_probability(concept_dist,image_drift_imp_l,num=2,labels=y_test))
    three_local_l_probs.append(local_imp_concepts_probability(concept_dist,image_drift_imp_l,num=3,labels=y_test))

    one_local_preds_l_probs.append(local_imp_concepts_probability(concept_dist,image_drift_imp_l,num=1,labels=localizer_bin_preds))
    two_local_preds_l_probs.append(local_imp_concepts_probability(concept_dist,image_drift_imp_l,num=2,labels=localizer_bin_preds))
    three_local_preds_l_probs.append(local_imp_concepts_probability(concept_dist,image_drift_imp_l,num=3,labels=localizer_bin_preds))

    reconstructed_single_concept = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=1)
    localizer_preds = localizer_model.l_predict(reconstructed_single_concept)
    reconstructed_single_concepts.append(accuracy_score(localizer_preds, y_test))
    reconstructed_single_concepts_preds.append(accuracy_score(localizer_preds, localizer_bin_preds))


    reconstructed_2_concept = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=2)
    localizer_preds = localizer_model.l_predict(reconstructed_2_concept)
    reconstructed_2_concepts.append(accuracy_score(localizer_preds, y_test))
    reconstructed_2_concepts_preds.append(accuracy_score(localizer_preds, localizer_bin_preds))

    reconstructed_3_concept = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=3)
    localizer_preds = localizer_model.l_predict(reconstructed_3_concept)
    reconstructed_3_concepts.append(accuracy_score(localizer_preds, y_test))
    reconstructed_3_concepts_preds.append(accuracy_score(localizer_preds, localizer_bin_preds))


    reconstructed_all_concept = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=21)
    localizer_preds = localizer_model.l_predict(reconstructed_all_concept)
    reconstructed_all_concepts.append(accuracy_score(localizer_preds, y_test))
    reconstructed_all_concepts_preds.append(accuracy_score(localizer_preds, localizer_bin_preds))

    print("Run:",j)





Fitting Unsupervised Craft....
Fitting Unsupervised Craft....
Fitting Craft....
Fitting Random Forest classifier...
Determine optimal parameters using cross validation
low threshold: 0.45 Mean:0.6085714285714285 High threshold:0.8, No. Leaves:20
Fitting complete.


Run: 0
Fitting Unsupervised Craft....
Fitting Unsupervised Craft....
Fitting Craft....
Fitting Random Forest classifier...
Determine optimal parameters using cross validation
low threshold: 0.25 Mean:0.4114285714285714 High threshold:0.6, No. Leaves:20
Fitting complete.
Run: 1
Fitting Unsupervised Craft....
Fitting Unsupervised Craft....
Fitting Craft....
Fitting Random Forest classifier...
Determine optimal parameters using cross validation
low threshold: 0.4 Mean:0.5828571428571429 High threshold:0.75, No. Leaves:20
Fitting complete.
Run: 2
Fitting Unsupervised Craft....


/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


Fitting Unsupervised Craft....
Fitting Craft....
Fitting Random Forest classifier...
Determine optimal parameters using cross validation
low threshold: 0.3 Mean:0.45714285714285713 High threshold:0.65, No. Leaves:20
Fitting complete.
Run: 3
Fitting Unsupervised Craft....
Fitting Unsupervised Craft....
Fitting Craft....
Fitting Random Forest classifier...
Determine optimal parameters using cross validation
low threshold: 0.2 Mean:0.39714285714285713 High threshold:0.6, No. Leaves:20
Fitting complete.
Run: 4
Fitting Unsupervised Craft....
Fitting Unsupervised Craft....
Fitting Craft....
Fitting Random Forest classifier...
Determine optimal parameters using cross validation
low threshold: 0.45 Mean:0.6228571428571429 High threshold:0.8, No. Leaves:20
Fitting complete.
Run: 5
Fitting Unsupervised Craft....


/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


...

Fitting Craft....
Fitting Random Forest classifier...
Determine optimal parameters using cross validation
low threshold: 0.2 Mean:0.37142857142857144 High threshold:0.55, No. Leaves:20
Fitting complete.
Run: 45
Fitting Unsupervised Craft....
Fitting Unsupervised Craft....
Fitting Craft....
Fitting Random Forest classifier...
Determine optimal parameters using cross validation
low threshold: 0.3 Mean:0.4857142857142857 High threshold:0.65, No. Leaves:20
Fitting complete.
Run: 46
Fitting Unsupervised Craft....
Fitting Unsupervised Craft....
Fitting Craft....
Fitting Random Forest classifier...
Determine optimal parameters using cross validation
low threshold: 0.35 Mean:0.5485714285714286 High threshold:0.75, No. Leaves:20
Fitting complete.
Run: 47
Fitting Unsupervised Craft....
Fitting Unsupervised Craft....


/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


Fitting Craft....
Fitting Random Forest classifier...
Determine optimal parameters using cross validation
low threshold: 0.4 Mean:0.6057142857142858 High threshold:0.8, No. Leaves:20
Fitting complete.
Run: 48
Fitting Unsupervised Craft....
Fitting Unsupervised Craft....


/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


Fitting Craft....
Fitting Random Forest classifier...
Determine optimal parameters using cross validation
low threshold: 0.3 Mean:0.4714285714285714 High threshold:0.65, No. Leaves:20
Fitting complete.
Run: 49


In [ ]:
import csv



methods = [ drift_localizer,
            one_local_one_global_l,
            one_local_l,
            two_local_l,
            three_local_l,
            one_local_l_probs,
            two_local_l_probs,
            three_local_l_probs,

            one_global_l,
            two_global_l,
            three_global_l,

            one_local_one_global_preds_l,
            one_local_preds_l,
            two_local_preds_l,
            three_local_preds_l,
            one_local_preds_l_probs,
            two_local_preds_l_probs,
            three_local_preds_l_probs,

            one_global_preds_l,
            two_global_preds_l,
            three_global_preds_l,

            reconstructed_single_concepts,
            reconstructed_single_concepts_preds,
            reconstructed_2_concepts,
            reconstructed_2_concepts_preds,
            reconstructed_3_concepts,
            reconstructed_3_concepts_preds,
            reconstructed_all_concepts,
            reconstructed_all_concepts_preds,

            label_maps,
            drift_ratios]

method_names = ["drift_localizer",
                "one_local_one_global_l",
                "one_local_l",
                "two_local_l",
                "three_local_l",
                "one_local_l_probs",
                "two_local_l_probs",
                "three_local_l_probs",

                "one_global_l",
                "two_global_l",
                "three_global_l",

                "one_local_one_global_preds_l",
                "one_local_preds_l",
                "two_local_preds_l",
                "three_local_preds_l",
                "one_local_preds_l_probs",
                "two_local_preds_l_probs",
                "three_local_preds_l_probs",

                "one_global_preds_l",
                "two_global_preds_l",
                "three_global_preds_l",
                "reconstructed_single_concepts",
                "reconstructed_single_concepts_preds",
                "reconstructed_2_concepts",
                "reconstructed_2_concepts_preds",
                "reconstructed_3_concepts",
                "reconstructed_3_concepts_preds",
                "reconstructed_all_concepts",
                "reconstructed_all_concepts_preds",
                "label_maps",
                "drift_ratios"]

# Write to CSV
with open('/content/drive/MyDrive/results/paper_experiment_repro.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    # writer.writerow(['Method'] + [f'Run_{i+1}' for i in range(50)])  # Header row
    writer.writerow(['Method'] + [f'Run_{i+1}' for i in range(50)])  # Header row
    for method, accuracies in zip(method_names, methods):
        writer.writerow([method] + accuracies)


In [ ]:
import pandas as pd
import numpy as np

# # Load CSV
# df = pd.read_csv('new_experiments_run6_l.csv')
# df = df.drop(['Method', axis=1)

# df = pd.read_csv('paper_experiment_D2.csv')
df = pd.read_csv('/content/drive/MyDrive/results/paper_experimentD2_repro.csv')
df = df.iloc[:29]

# Calculate mean and standard deviation
stats = {}
for method in df['Method']:
    accuracies = df[df['Method'] == method].drop('Method', axis=1).values.flatten().astype(float)
    mean = np.mean(accuracies)
    # median = np.median(accuracies)
    std = np.std(accuracies)
    # stats[method] = (mean, std, median)
    stats[method] = (mean, std)
# Example output for stats
print(stats)

{'drift_localizer': (np.float64(0.7844), np.float64(0.06535523442438766)), 'one_local_one_global_l': (np.float64(0.76), np.float64(0.07002221869600471)), 'one_local_l': (np.float64(0.7641333333333331), np.float64(0.06622372854364379)), 'two_local_l': (np.float64(0.7645333333333333), np.float64(0.06364400993023617)), 'three_local_l': (np.float64(0.7465333333333334), np.float64(0.06365867314566545)), 'one_local_l_probs': (np.float64(0.7611999999999999), np.float64(0.07245154702742149)), 'two_local_l_probs': (np.float64(0.7597333333333335), np.float64(0.06773753267822304)), 'three_local_l_probs': (np.float64(0.7498666666666668), np.float64(0.06657647232068298)), 'one_global_l': (np.float64(0.6624), np.float64(0.10688111775862626)), 'two_global_l': (np.float64(0.7170666666666665), np.float64(0.10153628798502425)), 'three_global_l': (np.float64(0.7346666666666667), np.float64(0.10324512364056501)), 'one_local_one_global_preds_l': (np.float64(0.8273333333333335), np.float64(0.092762540332231

In [ ]:
latex_table = """
\\begin{table}[h!]
\\centering
\\begin{tabular}{l|c}
\\hline
Method & Accuracy (Mean ± Std Dev) \\\\
\\hline
"""

for method, (mean, std) in stats.items():
    latex_table += f"{method} & {mean:.3f} ± {std:.3f} \\\\ \n"

latex_table += """
\\hline
\\end{tabular}
\\caption{Accuracy of different methods}
\\end{table}
"""

# Output the LaTeX table
print(latex_table)

##Model h tilde for paper is encompassed by "one_local_l_probs"

## We have other models here which use more concepts for possible future work


\begin{table}[h!]
\centering
\begin{tabular}{l|c}
\hline
Method & Accuracy (Mean ± Std Dev) \\
\hline
drift_localizer & 0.784 ± 0.065 \\ 
one_local_one_global_l & 0.760 ± 0.070 \\ 
one_local_l & 0.764 ± 0.066 \\ 
two_local_l & 0.765 ± 0.064 \\ 
three_local_l & 0.747 ± 0.064 \\ 
one_local_l_probs & 0.761 ± 0.072 \\ 
two_local_l_probs & 0.760 ± 0.068 \\ 
three_local_l_probs & 0.750 ± 0.067 \\ 
one_global_l & 0.662 ± 0.107 \\ 
two_global_l & 0.717 ± 0.102 \\ 
three_global_l & 0.735 ± 0.103 \\ 
one_local_one_global_preds_l & 0.827 ± 0.093 \\ 
one_local_preds_l & 0.851 ± 0.076 \\ 
two_local_preds_l & 0.845 ± 0.069 \\ 
three_local_preds_l & 0.825 ± 0.070 \\ 
one_local_preds_l_probs & 0.848 ± 0.086 \\ 
two_local_preds_l_probs & 0.843 ± 0.073 \\ 
three_local_preds_l_probs & 0.831 ± 0.069 \\ 
one_global_preds_l & 0.716 ± 0.133 \\ 
two_global_preds_l & 0.758 ± 0.132 \\ 
three_global_preds_l & 0.794 ± 0.121 \\ 
reconstructed_single_concepts & 0.752 ± 0.070 \\ 
reconstructed_single_concepts_preds